# Nhận dạng chữ số viết tay

**Lab 09 — Học máy ứng dụng**  
**Nguyễn Hoàng Hải — 2474802010099**

Huấn luyện MLP trên digits; khảo sát loss, tốc độ học, GridSearchCV và MLPRegressor cho giá nhà.

> Chạy các ô theo thứ tự từ thư mục gốc của dự án để các đường dẫn `data/...` được nhận diện. Cài các thư viện `numpy`, `pandas`, `matplotlib` và `scikit-learn` trước khi chạy.

In [ ]:
"""Nhan dang chu so viet tay va cac bai tap MLP tren digits/gia nha."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Chu so viet tay
X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
mlp_pipe = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=0),
).fit(X_tr, y_tr)
mlp = mlp_pipe[-1]
n_params = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
pred = mlp_pipe.predict(X_te)
print("Kich thuoc ma tran trong so:", [w.shape for w in mlp.coefs_])
print("Tong so tham so:", n_params, "| So vong hoc:", mlp.n_iter_)
print(f"Loss vong 1: {mlp.loss_curve_[0]:.4f}; vong 10: {mlp.loss_curve_[9]:.4f}; vong cuoi: {mlp.loss_curve_[-1]:.4f}")
print(f"Accuracy test: {accuracy_score(y_te, pred):.4f}; so anh doan sai: {(pred != y_te).sum()} tren {len(y_te)}")

plt.figure(figsize=(7, 4))
plt.plot(mlp.loss_curve_)
plt.xlabel("Vong hoc"); plt.ylabel("Loss"); plt.title("Loss tren tap hoc")
plt.grid(True); plt.tight_layout(); plt.show()

early = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000,
                  early_stopping=True, random_state=0),
).fit(X_tr, y_tr)
print("Dung som:", early[-1].n_iter_, "vong; accuracy:",
      f"{accuracy_score(y_te, early.predict(X_te)):.4f}")
other_models = {
    "Logistic co chuan hoa": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "KNN k=5": KNeighborsClassifier(),
    "SVM RBF": SVC(),
}
for name, model in other_models.items():
    print(name, f"{accuracy_score(y_te, model.fit(X_tr, y_tr).predict(X_te)):.4f}")

# Bai tap 4: toc do hoc va luu hinh toc_do_hoc.png
rates = [0.0001, 0.001, 0.01, 0.1]
rate_models = {}
print("\nBai tap 4: learning rate | so vong | loss cuoi | accuracy test")
for rate in rates:
    pipe = make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(100,), learning_rate_init=rate,
                      max_iter=1000, random_state=0),
    ).fit(X_tr, y_tr)
    model = pipe[-1]
    rate_models[rate] = model
    print(f"{rate:g} | {model.n_iter_} | {model.loss_curve_[-1]:.6f} | "
          f"{accuracy_score(y_te, pipe.predict(X_te)):.4f}")
plt.figure(figsize=(9, 5))
for rate, model in rate_models.items():
    plt.plot(model.loss_curve_, label=f"lr={rate}")
plt.xlabel("Vong hoc"); plt.ylabel("Loss"); plt.title("Anh huong cua toc do hoc")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.savefig("toc_do_hoc.png", dpi=150); plt.show()

# Bai tap 5: GridSearchCV
pipeline = make_pipeline(StandardScaler(), MLPClassifier(max_iter=1000, random_state=0))
param_grid = {
    "mlpclassifier__hidden_layer_sizes": [(10,), (50,), (100,), (50, 50)],
    "mlpclassifier__alpha": [0.0001, 0.01, 1],
}
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1).fit(X_tr, y_tr)
svm = make_pipeline(StandardScaler(), SVC()).fit(X_tr, y_tr)
mlp_grid_acc = accuracy_score(y_te, grid.predict(X_te))
svm_acc = accuracy_score(y_te, svm.predict(X_te))
print("\nBai tap 5:", grid.best_params_, "CV:", f"{grid.best_score_:.4f}")
print(f"MLP test: {mlp_grid_acc:.4f}; SVM test: {svm_acc:.4f}; MLP vuot SVM: {mlp_grid_acc > svm_acc}")

# Bai tap 6: hoi quy gia nha
houses = pd.read_csv("data/gia_nha.csv")
Xh = houses[["dien_tich", "so_phong", "tuoi_nha"]]
yh = houses["gia"]
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.25, random_state=17)
linear = LinearRegression().fit(Xh_tr, yh_tr)
reg = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(20,), max_iter=5000, random_state=0),
).fit(Xh_tr, yh_tr)
print("\nBai tap 6 MAE linear:", f"{mean_absolute_error(yh_te, linear.predict(Xh_te)):.4f}")
print("Bai tap 6 MAE MLP:", f"{mean_absolute_error(yh_te, reg.predict(Xh_te)):.4f}")
new_house = pd.DataFrame({"dien_tich": [250], "so_phong": [3], "tuoi_nha": [10]})
print("Du doan nha 250 m2 (3 phong, 10 nam): linear=",
      linear.predict(new_house)[0], "; MLP=", reg.predict(new_house)[0])
